<b><font size="6"> Cars 4 You: Predicting Car Values with ML </font></b><br><br>

`Group 40`

Ana Macedo (20250405)<br>
Catarina Mendinhas (20250422)<br>
Lourenço Silva (20250453)<br>
Maria Fonseca (20250380)<br>

## Abstract

This project aims to develop a predictive model capable of estimating car prices based on their characteristics, using the Cars 4 You dataset. The goal is to support the company’s evaluation process by introducing an automated solution that accelerates car assessments and improves overall efficiency.

The current phase of the project focuses on data exploration, including the identification of data quality issues, such as missing values, inconsistent string formats, and incorrect data types. A preliminary exploratory analysis was performed to understand the structure of the dataset, assess the relationships between features, and identify potential features influencing the car price.


## Import Libraries

The following libraries will help us develop the analyses and model for this project. Pandas and Numpy, provide the efficient tools for data manipulation, cleaning and numerical computations. Matplotlib and Seaborn are used to create clear and informative visualizations. Scikit-learn offers a range of Machine Learning tools for model training, spliting the data and evaluate model performance. Finally, os, math and ceil are imported to support file management and mathematical operations.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from math import ceil
import math

# For data manipulation and construction of a dataframe with different lists
import itertools 
from itertools import zip_longest

## Load the dataset
import os

# Set random seed for reproducibility
np.random.seed(40111) 

## Create Meta Data

Understanding the features helps interpret the data correctly and supports subsequent analysis and modeling steps. This section provides a brief description of each feature in the dataset, explaining its meaning and type.

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.



## Import Dataset

Here, we are importing with pd.read_csv() the files to start our project and converting them into Pandas DataFrames. CarID was set to index since it's the unique identifier in all data sets.

In [ ]:
sample = pd.read_csv('../data/sample_submission.csv')
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# EDA (Exploratory Data Analysis)

In this phase, we will explore the dataset to identify inconsistencies, missing values, duplicated values and potential relationships and patterns. This analysis will be very helpfull for the Preprocessing stage where we'll need to correct all these issues to improve the overall performance of our model.

### Define new index for our datasets

In [ ]:
sample.set_index('carID', inplace = True)
train.set_index('carID', inplace = True)
test.set_index('carID', inplace = True)

### Sample

In this first analysis, we look at the sample dataset given by the professor. This dataset includes the target feature, which shows how our final results should look.

In [ ]:
print("Initial Analysis of the Sample Submission Dataset")
print('')
print(sample.shape)
print('-----------------------')
print(sample.head())
print('-----------------------')
print(sample.tail())

In [ ]:
print("Continue to analyse the Sample Submission Dataset")
print('------------------------')
print('')
print(sample.info()) #From this output we can see that this dataset hasn't missing values
print('')
print('------------------------')
print(sample.describe())

## Initial Analysis Train/Test

In this section, we perform an initial exploration of the train and test datasets to understand their basic structure and content. We examine the first and last rows of the data, check general information about features, and review summary statistics. This helps us get a first overview and detect any immediate issues before deeper analysis.

In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
#Here we can see that the test shape is equal to the sample shape.

In [ ]:
train.head(10)

In [ ]:
train.tail(10)

From the visualization of the head and tail of the data base we can already understand that some errors exist:

    - Missing values
    - Values in the columns Year, hasDamage, previousOwners that should be integers as floats (2020.0)
    - Floats on mpg, previousOwners column with diferent decimal cases length.
    - It looks like the column hasDamage only contains 0's. It's probably a boolean.
    - Negative values in columns that shouldn't have, per example in previousOwners column.
We will further analyse this using describe and info.

It's also possible to see that some strings have the same information written in different forms (Petrol as etrol).
To solve this problem we will uniformize all the values in data preparation

In [ ]:
train.info()

From info we can see that:

    - the year values as floats is impossible.
    - The previousOwners, hasDamage also as floats but they should be integers and booleans respectively.
    - Missing values in all features except the price.

What will we do?

    Analyse with describe to have a different view

In [ ]:
train.describe()

From the numeric describe we can see that we have some weird values:

    1. negative mileage, tax, mpg, engineSize, previousOwners in the minimum value
    2. hasDamage is a boolean but we can see that instead of 0 and 1 we only have 0 and Nones*
    3. previousOwner has a float? Should we round it?

*check in the hasDamage column

What will we do:

    1. Count the number of negative values and decide if we should drop or change them.
    2. Replace the nones by 1's. (data-preparation)
    3. Count the number of float values and decide to drop or round them.

In [ ]:
train.describe(include='object').T

From the categorical describe, we can see that some columns contain missing values, indicating incomplete data that may need to be addressed before modeling. Additionally, we can observe the number of unique categories in each column, as well as the most frequent category and its frequency. This information gives us an initial understanding of the distribution and variability of the categorical features, which can help guide preprocessing decisions such as encoding or handling rare categories.


### **Missing Values**

In this part, we examine the dataset to identify any missing values in the features. Understanding where data is missing helps us plan how to handle it, whether through imputation, removal, or other preprocessing steps, ensuring the dataset is suitable for modeling.

In [ ]:
# Number of rows in the dataset
rows_number = train.shape[0]

# Number of missing values per column
missing_counts = train.isnull().sum()

# Percentage of missing values per column
missing_percentage = (missing_counts / rows_number) * 100

# Show in DataFrame format, sorted from highest to lowest
missing_df = missing_percentage.sort_values(ascending=False).reset_index()
missing_df.columns = ['Feature', 'Missing_Percent']

print(missing_df)

Although some features have missing values, the percentage of missing data is relatively low across all columns. This information highlights which features may require attention during preprocessing.

After this analysis, we decided to save this information in a separate DataFrame to use later during preprocessing. This allows us to have a clear reference of incomplete data and helps streamline the cleaning and imputation process.

In [ ]:
# Go one level up from the notebooks folder to reach the repo root
processed_dir = "../EDA_outputs"
os.makedirs(processed_dir, exist_ok=True)

# Save the missing values summary to a CSV file
missing_df.to_csv(f"{processed_dir}/missing_values_summary.csv", index=True)

### **Duplicates**

Here, we check for duplicate entries in the dataset. Detecting and removing duplicates is important to prevent bias and ensure that each observation contributes uniquely to the analysis and model training.

In [ ]:
print(f"The duplicates without the index is: {train.duplicated().sum()}") 
print(f"The percentages of duplicates is: {(train.duplicated().sum()/len(train) * 100).round(4)}%")

It's also important to check for duplicates in the index, because assign the feature carID as our index.

In [ ]:
print(f"The duplicates in carID (defined as index) is: {train.index.has_duplicates}")

From the code above, we can conclude that there are four entries with duplicated values but different IDs. Since the percentage of duplicates in the dataset is 0.0053% and there are 0% duplicates in the index, we decided to remove the duplicate records found in the dataset.

In [ ]:
train = train.drop_duplicates()

## Univariate Analysis

In this section, we analyze each feature individually to understand its distribution and main characteristics. This helps us identify patterns, outliers, and possible data quality issues within each feature.

### Unique Values

We examine the number of unique values in each feature. This helps us understand the diversity of the data, identify categorical features, and detect features with little or no variability that may be less informative for modeling.

In [ ]:
for column in train.columns:
    unique_values = train[column].unique()
    print(f"Unique values on the column '{column}': {unique_values}")
    print('-----------------------------------')
    print('')


Comments on the results above for each feature:
- `Brand` have significant inconsistencies with case sensitivity, ('mercede', 'FOR'), and fragmented entries ('pel', 'udi').

- `Model` have a lot of variations due to leading/trailing spaces, inconsistent casing, typos ('Focu' instead of 'Focus'), and incomplete entries ('A Clas').

- `Year` have decimal/fractional values (e.g., 2023.36707842) suggests data encoding errors or floating-point artifacts. 

- `Price` have numeric and diverse.

- `Transmission` column with multiple variants and typos exist for common types (e.g., 'Manual', 'manua', 'anual'). Also we noticed that in this columns there are 'Unknown' values.

- `Mileage` appears mostly clean and numeric.

- `FuelType` have many spelling and case variations (e.g., 'Petrol', 'petrol', 'etrol') exist.

- `Tax` with negative and fractional values are suspicious. 

- `MPG` looks like contain outliers and invalid negative values.

- `EngineSize` includes some unusual negative and zero values that should be verified.

- `PaintQuality%` have some values greater than 100%, what is not possible.

- `PreviousOwners` contains unexpected negative/fractional values and NaNs.

- `HasDamage` is Binary flag with values 0 and NaN.

### Count Values

Here, we analyze the frequency of each value in the categorical features. Studying value counts allows us to see which categories are most common, identify imbalanced distributions, and detect rare or unusual values that may require special handling during preprocessing.

In [ ]:
for column in train.select_dtypes(include=['number']).columns:
    count_values = train[column].value_counts()
    print(f"The number of the distinct values in the column '{column}' is: {count_values}")
    print('-----------------------------------')
    print('')

Most of all the comments that can be done from the results above were already done after the .unique() analysis. 
With both this analysis we are planning to do the following for each feature:

- `Brand`, `Model`, `Transmission` and `FuelType`: unify capitalization, spacing and correct spelling errors and inconsistent spellings.

- `Year`: convert all values to integers and fix or remove non-sensical decimal values.

- `Price`: drop from the dataset and assign to a target feature.

- `Mileage`, `Tax`, `MPG`, `EngineSize` and `PreviousOwners`: ensure values are numeric and postive, detect and handle unrealistic or extreme values (e.g., negative, float or abnormall values in the feature context).

- `PaintQuality%`: cap values between 0 and 100.

- `HasDamage`: maybe assign the nan values to 1 cause we only have 0's and this should be a boolean feature.

Besides all this notes we need also to consider all the missing value noticed before on the DataFrame.


### Categories Frequency

Here we examine the categorical features in the dataset by computing their frequency distributions. This allows us to identify the most common categories, detect rare or unbalanced classes, and better understand the overall composition of each feature. 

In [ ]:
for column in train.select_dtypes(include=['object']).columns:
    vc = train[column].value_counts(dropna=False)
    print(f"--- {column} (n_unique={vc.shape[0]}) ---")
    display(vc.head(10))


### Target feature (Price)

In this subsection, we explore the characteristics of the target feature, Price. Since price exhibit a right-skewed distribution with a wide range of values, we also analyze the logarithm of Price (log(Price)). Using the logarithmic transformation helps stabilize variance, reduce the influence of extreme values, and allows for a more interpretable analysis in terms of relative or percentage changes rather than absolute differences. This approach is particularly useful for modeling and understanding economic or financial features that typically follow multiplicative patterns.

In [ ]:
# Distribution & Transform
train['price_log'] = np.log1p(train['price'])

# Statistics
# Calculate skewness, which measures how asymmetric the data distribution is
print("Price skew:", train['price'].skew().round(3)) 
print("Log(price) skew:", train['price_log'].skew().round(3))

# Plots
fig, axes = plt.subplots(1,3, figsize=(14,5))
sns.histplot(train['price'], ax=axes[0], kde=True)

axes[0].set_title('Price distribution')
sns.boxplot(x=train['price'], ax=axes[1])
axes[1].set_title('Price boxplot')

# Also plot log-price
sns.histplot(train['price_log'], ax=axes[2], kde=True)
axes[2].set_title('Log1p(Price) distribution')

plt.tight_layout()
plt.show()

The distribution of Price is highly right-skewed, with most observations concentrated at lower values and a long tail of high prices, as shown in both the histogram and boxplot. This indicates the presence of several extreme values (outliers) and a non-normal distribution, which could negatively affect modeling performance if used directly.

With the application the logarithmic transformation, the distribution becomes much more symmetric and closer to a normal shape. This transformation reduces the impact of extreme values, stabilizes the variance, and facilitates the use of linear models and other statistical methods that assume normality. Additionally, it allows for interpreting model results in terms of percentage changes rather than absolute price differences, which is more meaningful in most economic and financial contexts.

In [ ]:
cat_cols = train.select_dtypes(include=['object','category']).columns.tolist()
for c in cat_cols:
    vc = train[c].value_counts(dropna=False)
    print(f"--- {c} (n_unique={vc.shape[0]}) ---")
    display(vc.head(10))

### Descriptive Statistics and Visualizations

This section starts by split our data in metric, discrete and continuous, and non-metric features and after that we summarize the main characteristics of the dataset using descriptive statistics and visualizations. This helps us understand distributions, trends, and potential anomalies, providing insights that guide further analysis and modeling decisions.

In [ ]:
train

In [ ]:
train

In [ ]:
# Separate the features
metric_features = ['year', 'price', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners']
non_metric_features = train.drop(columns=metric_features).columns.tolist()

continuous_features = ['price', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%']
discrete_features = ['year', 'previousOwners', 'paintQuality%']

# Organize the new lists in a table to better visualize them
data = list(zip_longest(
    metric_features,
    non_metric_features,
    continuous_features,
    discrete_features,
    fillvalue=""
))

headers = [
    "Metric Features",
    "Non-Metric Features",
    "Continuous Features",
    "Discrete Features"
]

df_table = pd.DataFrame(data, columns=headers)
print(df_table.to_string(index=False))


#### **Histograms**

Are used to examine the distribution of continuous features. Histograms help us see the shape of the data, detect skewness, and identify potential outliers or patterns in the feature values.

In [ ]:
# General settings
sns.set(style="whitegrid", palette="pastel", font_scale=1.3)

sp_rows, sp_cols = 3, 2
fig, axes = plt.subplots(sp_rows, sp_cols, figsize=(24, 15))

# Plot data
# Iterate across axes objects and associate each histogram
for ax, feat in zip(axes.flatten(), continuous_features):
    kde_flag = True
    sns.histplot(data=train, x=feat, bins=15, kde=kde_flag, ax=ax)
    ax.set_title(feat, fontsize=13, pad=10)
    ax.set_ylabel("Frequency")
    ax.grid(True, linestyle="--", alpha=0.6)

# General Layout
fig.suptitle("Distribution of Continuos features", fontsize=18, fontweight='bold', y=0.95)
fig.tight_layout(rect=[0, 0, 1, 0.96])

plt.show()


With the evaluation of the histograms we can affirmate that:

`Price` - The distribution is strongly right-skewed, most cars are low or mid-priced, but a few luxury vehicles inflate the tail. Some prices are extreme outliers, possibly due to rare high-end models. From here we can say that the market is dominated by affordable vehicles.

`Mileage` - Also right-skewed, which is expected. Most used cars have moderate mileage, and a few have very high values. The histogram shows again negative values.

`Tax` - Sharp peak around a specific value (≈150–200), indicating standardized taxation rates for most vehicles. Some negative or zero values are also noticed.

`Mpg` - Distribution shows that most cars cluster around 20 to 90 mileage per gallon , with few very high or very low values, the negatives values being incorrect.

`EngineSize` - Multiple peaks, possibly due to different engine categories (1.0L, 1.5L, 2.0L, etc.). The pattern may reflect distinct market segments.

`PaintQuality%` - Fairly uniform distribution across mid-to-high values (40–100%), indicating most vehicles have decent paint condition. We can see a few very low or very high values that are outliers. The highest outliers are incorrect values because this column should have values between 0 and 100 since it's a percentage. 

#### **BarPlots**

Are used to examine the distribution of discrete features. These graphs, also help us see the shape of the data, detect skewness, and identify potential outliers or patterns in the feature values.

In [ ]:
# General settings
sns.set(style="whitegrid", palette="pastel", font_scale=1.3)

sp_rows, sp_cols = 1, 2
fig, axes = plt.subplots(sp_rows, sp_cols, figsize=(24, 12))

# Plot data
for ax, feat in zip(axes.flatten(), discrete_features):
    sns.countplot(data=train, x=feat, ax=ax, color='skyblue', edgecolor='black')
    ax.set_title(f"{feat}", fontsize=13, pad=10)
    ax.set_ylabel("Frequency")
    ax.set_xlabel(feat)
    ax.grid(True, linestyle="--", alpha=0.6)

fig.suptitle("Distribution of Discrete features", fontsize=18, fontweight='bold', y=0.95)
fig.tight_layout(rect=[0, 0, 1, 0.96])

plt.show()


#### **Boxplots** (Outliers Detection)

Are used to visualize the spread and variability of numerical features. Boxplots highlight the median, quartiles, and outliers, making it easier to detect unusual values and understand the overall data distribution.

In [ ]:
# Style settings
sns.set_theme(style="whitegrid", palette="pastel")

# Prepare figure with appropriate number of rows and columns
n_features = len(metric_features)
n_cols = 2  
n_rows = ceil(n_features / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

# Plot boxplots
for i, feat in enumerate(metric_features):
    ax = axes[i]

    # Boxplot with highlighted median line
    sns.boxplot(
        x=train[metric_features][feat],
        ax=ax,
        color='skyblue',
        medianprops={"color": "darkblue", "linewidth": 2},
        boxprops={"alpha": 0.7}
    )
    
    ax.set_title(f"{feat}", fontsize=12, fontweight='bold')
    ax.set_xlabel("")  # Remove automatic x-label
    ax.grid(True, linestyle="--", alpha=0.4)

# Remove empty axes (if the number of features is not a multiple of n_cols)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# General title and layout
plt.suptitle("Boxplots of Metric Features", fontsize=18, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


Based on the boxplots shown above, it can be observed that:

`Price`- This boxplot is right-skewed, with a large proportion of outliers ranging approximately from £40,000 up to £160,000, showing a wide interval compared to the main body of the data.

`Mileage` - Mileage displays a positively skewed distribution with several high-value outliers and a negative value appears in the data.

`Tax`- The distribution is concentrated around specific values but includes some outliers.

`Mpg`- The distribution is mostly concentrated but includes both low and very high outliers. Some extreme values, such us those above 200 mpg are unrealistic and will be validated.

`EngineSize`- The boxplot shows most engines concentrated between 1.0 and 2.0L, with several larger values appearing as outliers.

`PaintQuality%`- This boxplot shows few distinct values spread across a wide range, with numeric codes representing quality levels.

`PreviousOwners`- The distribution is concentrated between 1 and 3 owners, with a few outliers, including cars with more than 6 owners and some negative values.

`Year` - The distribution is left-skewed, with a wide range of outliers from the middle 1990's until early 2000's.

## Bivariate Analysis

Here we study the relationship between two features at a time, mainly between the features and the target feature. This analysis helps us understand how each feature might influence the target and whether any correlations exist.

### GroupBy

We use groupby operations to summarize and compare data across some different categories. This allows us to examine how numerical features vary between groups, identify patterns, and explore relationships that may be important for modeling.

In [ ]:
print(train.groupby('hasDamage')['price'].mean())

print('--------------------------------------------------------')

print(train.groupby('year')['price'].mean())

print('--------------------------------------------------------')

print(train.groupby('previousOwners')['price'].mean())

print('--------------------------------------------------------')

print(train.groupby('Brand')['price'].mean())


The mean of price of the cars without damage is approximately 16883. It would be interesting to compare with the one's with damage (1's), but we don't have any information about them.

It's also possible to verify that the price varies depending on the year of manufacture of the cars. This is, the older the car, the cheaper it is.

For the number of previous Owners, we don't see major differences on the average price. 

Finally, when analysing the average price per brand, we can already see some differences between brands, per example, Hyundai's cars have lower average than Audi's cars.


### Correlation Between Features

This section is quite important. We analyze the relationships between numerical features using correlation measures. This helps us identify which featires are strongly related, detect potential multicollinearity, and understand how features may influence each other or the target feature.

In [ ]:
numeric_features = train[['year', 'mileage', 'price', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners']]
numeric_features.corr(method="pearson").round(3)

From the code before it's difficult to get conclusions. We will visualize this matrix using a heatmap. The heatmap provides an intuitive overview of the strength and direction of relationships, making it easier to identify strongly correlated features and patterns in the data.

Here we use heatmaps based on Pearson and Spearman correlation coefficients. Pearson correlation measures linear relationships, while Spearman correlation captures monotonic relationships. 

In [ ]:
corr = train[metric_features].corr(method="pearson").round(2)

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Visualize correlation matrix
fig = plt.figure(figsize=(10, 8))

sns.heatmap(
    corr,
    mask=mask,                # hide upper triangle
    annot=True,               # show values
    cmap="coolwarm",          # divergent color map
    center=0,                 # center colormap in 0
    linewidths=0.5,           # lines between cells to help visualization
    vmin=-1, vmax=1,          # fix scale
    square=True               # make cells square-shaped
)


plt.title("Correlation Matrix (Pearson)", fontsize=14, pad=15)
plt.tight_layout() # improve layout by reducing overlaps
plt.show()


In [ ]:
corr = train[metric_features].corr(method="spearman").round(2)

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Visualize correlation matrix
fig = plt.figure(figsize=(10, 8))

sns.heatmap(
    corr,
    mask=mask,                # hide upper triangle
    annot=True,               # show values
    cmap="coolwarm",          # divergent color map
    center=0,                 # center colormap in 0
    linewidths=0.5,           # lines between cells to help visualization
    vmin=-1, vmax=1,          # fix scale
    square=True               # make cells square-shaped
)


plt.title("Correlation Matrix (Spearman)", fontsize=14, pad=15)
plt.tight_layout() # improve layout by reducing overlaps
plt.show()

So from the correlation analysis with both spearman and pearson methods we can say that our model’s most important numerical predictors, for our target price, will likely be:
- year, mileage, engineSize
- tax and mpg might add secondary predictive power.
- paintQuality%, previousOwners, and hasDamage probably won’t help much unless cleaned or reinterpreted.

### Pairplot 

In [ ]:
sns.pairplot(train[continuous_features])

## Multivariate Analysis

In this part, we explore the interactions between several features simultaneously. The goal is to identify more complex relationships and combined effects that may not be visible when analyzing features separately.

### 3D Scatterplot

The 3D scatter plot visualizes the relationship between a car’s price, year of registration, and mileage.
Each point represents a car, where newer models (higher year values) and those with lower mileage tend to have higher prices.
This graph helps identify depreciation patterns — as mileage increases or the car becomes older, its value generally decreases.
It provides a clear, three-dimensional view of how these features interact to influence car pricing.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(train['year'], train['mileage'], train['price'], c=train['price'], cmap='viridis')
ax.set_xlabel('Year')
ax.set_ylabel('Mileage')
ax.set_zlabel('Price (£)')

plt.title('3D Scatter Plot of Year, Mileage, and Price')
plt.tight_layout()
plt.show()


The three-dimensional analysis of Year, Mileage, and Price demonstrates that vehicle depreciation follows a cone shaped pattern, with recent models (2015-2023) and low mileage concentrated at the top of the price spectrum, while older vehicles converge toward a minimum value floor regardless of accumulated use. The pattern reveals three distinct depreciation dynamics: extreme sensitivity to mileage in recent vehicles, moderate depreciation in intermediate-age cars, and price stagnation in older models. Crucially, Year and Mileage do not act independently but through a multiplicative interaction, where the impact of mileage is amplified in newer cars and negligible in older ones, explaining the limitation of simple linear models and suggesting that interaction features and logarithmic transformations would be necessary to improve predictive accuracy, as well as investigation of outliers that indicate qualitative factors not captured by the model.

### Scatter Matrix

Illustrates the relationships between continuous features (price, year, mileage, engine size, and mpg) while distinguishing cars by fuel type.
Each scatter plot shows how two features interact, and the diagonal plots display their individual distributions.
From this visualization, we can identify patterns such as newer cars with lower mileage tending to have higher prices, or how fuel type influences fuel efficiency (mpg) and engine size.
Overall, the pairplot provides a comprehensive multivariate overview of how these core features relate to each other across different fuel categories.

In [ ]:
sns.pairplot(train, vars=['price', 'year', 'mileage', 'engineSize', 'mpg'], hue='fuelType')
plt.show()

The pairplot reveals strong relationships between vehicle price and features such as year, mileage, and engine size. It shows that newer cars with lower mileage tend to have higher prices, while vehicles with larger engines are generally more expensive but less fuel-efficient (lower mpg). There is also a clear negative correlation between engine size and fuel efficiency, indicating that bigger engines consume more fuel. The fuelType feature shows a predominance of petrol cars, although multiple variations in spelling suggest the need for data standardization. Overall, the dataset displays expected patterns of vehicle valuation but also contains outliers and inconsistencies that should be addressed for more accurate analysis.

## End of the Notebook

From our exploratory data analysis, we gained a better understanding of the different datasets. We observed key characteristics of the features, including distributions, ranges, and the presence of any outliers. The correlation analysis revealed some notable relationships between features, suggesting which features might be most relevant to the target. Based on these observations, we can form initial hypotheses about which features may influence the target and guide the next steps in feature selection and model building..